# Multi-hop questions with `QueryPlanEngine`

`QueryPlanEngine` wraps any other engine. Given a question that cannot be answered
from one retrieval, it:

1. decomposes the question into a DAG of subqueries,
2. answers each subquery through the wrapped engine, in dependency order,
3. rewrites dependent subqueries so they are self-contained, injecting the answers
   they depend on,
4. returns the answer of the plan's final (sink) subquery.

The wrapped engine never learns it is being planned for; it just receives ordinary
queries. So the planner composes with local, global, naive or mix search
interchangeably.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    QueryPlanEngine,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")

# Two-hop questions
QUESTIONS = [
    "Who created the C programming language, and where did that person's father work?",
    "What did the person who died on October 12, 2011 create, and who did they work with?",
]

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/query_plan_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Build the graph

The expensive cell. Run once.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

In [ ]:
base_engine = LocalSearchEngine(
    llm=llm, 
    knowledge_graph=knowledge_graph, 
    embedder=embedder
)
planner = QueryPlanEngine(base_engine)

# Parameters are forwarded verbatim to the wrapped engine for every subquery.
params = LocalParams(top_k=10)

## Inspect the plan

`process_query` decomposes without executing, which is how you debug a planner
that produces the wrong hops.

In [ ]:
for subquery in await planner.process_query(QUESTIONS[0]):
    depends = ", ".join(subquery.depends_on) or "-"
    print(f"{subquery.id} intent={subquery.intent} depends_on={depends}")
    print(f"    {subquery.query}")

## Answer with the planner

Batched: subqueries from different top-level questions that become ready at the
same step are answered in one call.

In [ ]:
for response in await planner.batch_query(QUESTIONS, params):
    print(f"Q: {response.query}")
    print(f"A: {response.response}\n")

In [ ]:
await knowledge_graph.index.close()